| Model | Ligand inputs | Total encoded inputs in LOLO / IID |
| --- | --- | --- |
| `ligand_ohe` | Ligand one-hot encoding | 35 / 36 |
| `selected_5` | Boltzmann-average, minimum and range of buried volume; dipole; HOMO-LUMO gap | 33 / 33 |
| `selected_2` | Boltzmann-average and minimum buried volume | 30 / 30 |
| `pc_top` | Top 3 original descriptors from each of PC1-PC4, deduplicated | 40 / 40 (12 ligand descriptors in this snapshot) |
| `pc_scores` | Each of the 190 reference descriptors times its loading in each of PC1-PC4, unsummed | 788 / 788 |

| Evaluation | Definition | Fits across five models |
| --- | --- | ---: |
| LOLO / OOD | Train on seven ligands, test all rows of the eighth; all eight ligands | 40 |
| Matched IID | The same eight-fold geometry with random membership: folds sized to each LOLO test set, every row tested exactly once, all ligands in every fold | 40 |
| Five-fold | `KFold(5, shuffle=True, random_state=42)` | 25 |
| 80/20 | `train_test_split(test_size=0.2, random_state=42)` | 5 |
| LOLO reporting | Individual ligands, mean across ligand-left-out models, pooled after the eight folds; reuse the same fits | 0 additional |

Production is **110 fits**, each with five optimizer restarts. Prepare locally, inspect the results, then click the final export button. All utility code lives in `hazel_gp/`. No model training occurs in this notebook.

`pc_scores` keeps every descriptor at its own PCA weight rather than collapsing them into four coordinates; set `features.pc_scores_form` to `component_scores` in the config for the earlier four summed scores.

### Settings
Open this notebook from the extracted project folder. Change paths or settings here before preparation; saved bundles include the resolved settings.

In [3]:
from pathlib import Path
import pandas as pd
import os
from IPython.display import display
from hazel_gp.config import load_config
from hazel_gp.data import download_sources, prepare_data
from hazel_gp.features import model_summary
from hazel_gp.splits import evaluation_summary

ROOT = Path.cwd().resolve()
assert (ROOT / "hazel_gp").is_dir(), "Open the extracted project folder in VS Code first."
CONFIG = ROOT / "configs/default.json"
cfg = load_config(CONFIG)
# Optional new version folders:
# cfg["paths"]["raw"] = "data_hazel/raw_v2"
# cfg["paths"]["prepared"] = "data_hazel/prepared/group1_v2"
display(model_summary(cfg))
display(evaluation_summary(cfg))

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

### Download the original data
These are the three original notebook URLs. Cached files are reused and hash-checked; no changes are made to your old `data/` directory.

In [ ]:
sources = download_sources(cfg, ROOT)
import pandas as pd
display(pd.DataFrame(sources).T[["filename", "url", "sha256"]])

### Clean and audit
The UV-area response is the target. Exclusion reasons, mappings and all row counts are available before export.

In [ ]:
prepared = prepare_data(cfg, ROOT)
display(pd.Series(prepared.audit, name="audit"))
display(prepared.mapping)
display(prepared.excluded["exclusion_reason"].value_counts())
display(prepared.reactions.head())

### Review the exact models and splits
PCA uses every ligand in the original linked Kraken descriptor reference. Its fitted state is frozen; reaction scalers are fitted inside the training folds on Hazel.

`ligand_columns_into_gp` counts the columns the GP receives, and `descriptors_behind_those_columns` says how many descriptors produced them. With the default `pc_scores_form = "loading_weighted"`, `pc_scores` performs no dimensional reduction: input `PC{i}_x_{descriptor}` is that descriptor's fold-standardized value times its loading in component *i*, giving 190 x 4 = 760 ligand inputs. Summing one component's 190 columns reproduces that component's score, so nothing the scores are built from is discarded. `pc_top` is the opposite; it keeps three raw descriptors per PC and drops the other 178. The loading summary reports how much of each PC's squared loading length those three carry, and the loading table lists the individual per-descriptor weights.

The feature grid names each encoded input per model, ligand features first, then the shared categorical block that is held constant across all five models. It shows the first `MAX_FEATURES_SHOWN` names with a totals row; the exported bundle records every name in `model_features.csv`. Categories come from all prepared rows, so this is the matched-IID column set: a LOLO training fold drops the held-out ligand's one-hot column.

In [ ]:
from hazel_gp.features import model_feature_table, pca_loading_summary, pca_loading_table

MAX_FEATURES_SHOWN = 25  # Raise to list every encoded input; the bundle always stores the full list.
display(model_summary(cfg, prepared.reference))
display(model_feature_table(prepared, max_features=MAX_FEATURES_SHOWN))
display(evaluation_summary(cfg, prepared.tasks))
display(pd.DataFrame(prepared.split_entries))
display(pd.Series(prepared.reference.variance_ratio, index=["PC1", "PC2", "PC3", "PC4"], name="explained_variance_fraction"))
display(pd.DataFrame(prepared.reference.top_by_pc))
# Each pc_scores input is the full loading vector applied to every reference descriptor.
display(pca_loading_summary(prepared.reference))
display(pca_loading_table(prepared.reference, top=MAX_FEATURES_SHOWN))

### Final export
Edit both destinations if they already exist. Click **Export reviewed inputs** once to write the prepared data and upload ZIP. Upload/extract that ZIP on Hazel and follow README steps 5-7. No training jobs are submitted here.

In [ ]:
from hazel_gp.notebook import export_controls
display(export_controls(prepared, ROOT))

For a non-widget export, use `export_prepared` and `create_upload_archive` as documented in the README. They use the same validation and refuse to overwrite existing bundles.